In [1]:
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup as bs
import requests
from tqdm import tqdm
from time import sleep
import json

In [2]:
def get_top_elo(date=None):
    top_teams = pd.DataFrame()

    fir = ['team','elo_(k=32)','elo_(k=64)','glicko_1','glicko_2']
    sec = ['', 'avg_7d', 'δ7d', 'avg_30d', 'δ30d', 'rating', 'μ', 'σ', 'δrat.7d']
    
    url = f"https://www.datdota.com/ratings"
    page = requests.get(url)

    soup = bs(page.text,'html.parser')
    table_body = soup.find('table')

    row_data = []
    for row in table_body.find_all('tr'):
        col = row.find_all('td')
        col = [ele.text.strip() for ele in col]
        row_data.append(col)

    idx = pd.MultiIndex(levels=[fir,sec],codes=[[0,1,1,1,1,1,2,2,2,2,2,3,3,3,3,4,4,4,4],[0,5,1,2,3,4,5,1,2,3,4,5,6,7,8,5,6,7,8]], names=['lvl1','lvl2'])

    top_teams = pd.DataFrame(row_data[2:],columns=idx)

    top_teams['team'] = top_teams['team'].apply(lambda x: x.split('\n')[0])
    if date:
        top_teams['date'] = date

    #Flatten multiindex
    top_teams.columns = ['_'.join(x) for x in top_teams.columns.to_flat_index()]
    
    #Fix team column
    top_teams.rename({'team_':'team'},axis=1,inplace=True)
    
    return top_teams

In [3]:
df = get_top_elo()

In [4]:
def get_teams(pages=1):
    df = pd.DataFrame()
    for page in tqdm(range(pages)):
        request = requests.request('GET',f"https://api.opendota.com/api/teams?page={page}")
        data = request.json()
        df = pd.concat([df,pd.DataFrame(data)],ignore_index=True)
        sleep(1)
    df['team_id'] = df['team_id'].astype(int)
    return df

In [5]:
teams = get_teams(15)

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:24<00:00,  1.62s/it]


In [27]:
test = pd.merge(df,teams,how='left',left_on='team',right_on='name')
test = test.dropna()
test['team_id'] = test['team_id'].astype(int)

In [28]:
#test[test['name'].isna()]
test[['team','team_id']]

,team,team_id
0,PSG.LGD,15
1,PSG.LGD,7566630
2,PSG.LGD,6658355
3,PSG.LGD,8125994
4,Team Aster,6209166
...,...,...
136,Eternity,8260932
137,Simply TOOBASED,8272699
138,Arkosh Gaming,8180753
139,felt,8261882


In [29]:
def get_team_players(team_id,current=None):
    request = requests.request('GET',f"https://api.opendota.com/api/teams/{team_id}/players")
    data = request.json()
    df = pd.DataFrame(data)
    if current:
        df = df[df['is_current_team_member']==True]
    
    return df

In [30]:
def get_all_team_players(team_id,current=None):
    all_players = pd.DataFrame()
    for team in tqdm(team_id,total=len(team_id)):
        player = get_team_players(team,current)
        player['team_id'] = team
        all_players = pd.concat([all_players, player],ignore_index=True)
        sleep(1)
    
    return all_players

In [31]:
list(test['team_id'].values)

[15,
 7566630,
 6658355,
 8125994,
 6209166,
 8488432,
 8488438,
 7119388,
 2586976,
 7266740,
 5039673,
 6519957,
 8724984,
 6209804,
 8488480,
 8605863,
 8291895,
 7732977,
 2163,
 8261500,
 1838315,
 8260983,
 39,
 7172615,
 5,
 350190,
 7714035,
 8599101,
 7391077,
 5222250,
 8254400,
 7441136,
 7079109,
 8721219,
 7390454,
 4,
 8488462,
 7554697,
 726228,
 36,
 8597976,
 7424172,
 8214850,
 7453020,
 8728920,
 1520578,
 7422789,
 1846548,
 7375761,
 7422041,
 8126892,
 8244493,
 8687717,
 8668460,
 8701453,
 8376696,
 8525778,
 8254145,
 7408845,
 4189016,
 7356881,
 7119077,
 8578859,
 5029857,
 8376426,
 2672298,
 8124688,
 8488449,
 8572181,
 8131728,
 1798457,
 8255888,
 8390848,
 8384158,
 8070280,
 8598715,
 8616073,
 5838173,
 111474,
 2108395,
 8261554,
 8112124,
 8375259,
 8169775,
 5014799,
 8261648,
 7407260,
 8741674,
 8686786,
 8597391,
 8255756,
 8598633,
 8588969,
 8118197,
 7586995,
 1061269,
 8680612,
 8620831,
 8722443,
 8545488,
 8604954,
 8271508,
 8263304,
 87

In [32]:
players = get_all_team_players(pd.Series(test['team_id'].values),True)

100%|████████████████████████████████████████████████████████████████████████████████| 129/129 [03:32<00:00,  1.65s/it]


In [33]:
players.account_id.nunique()

482

In [34]:
players

,account_id,name,games_played,wins,is_current_team_member,team_id
0,898754153,萧瑟,499,350,True,15
1,111114687,y`,484,338,True,15
2,157475523,XinQ,484,338,True,15
3,118134220,Faith_bian,482,338,True,15
4,173978074,NothingToSay,443,311,True,15
...,...,...,...,...,...,...
477,188423794,lies,88,25,True,8230115
478,113055105,Bolo,61,27,True,8230115
479,116080282,Argius,53,21,True,8230115
480,417512732,feelinG♥ :3,53,21,True,8230115


In [35]:
players.to_csv('top_players.csv',index=False)

In [36]:
COLUMNS = ['Match']

def get_player_matches(account_ids):
    all_matches = pd.DataFrame(columns=COLUMNS)
    errors = []
    for account in tqdm(account_ids,total=len(account_ids)):
        try:
            response = requests.request('GET',f"https://www.datdota.com/players/{account}")
            soup = bs(response.content, "html.parser")
            table = soup.find_all('table')[0] # Find the first "table" tag in the page
            rows = table.find_all("tr")
            cy_data = []
            for row in rows:
                cells = row.find_all("td")[:1] # Get list of Most recent 100 Match Ids
                cy_data.append([cell.text for cell in cells]) # For each "td" tag, get the text inside it
            cy_data = pd.DataFrame(cy_data,columns=COLUMNS)
            cy_data = cy_data.loc[cy_data['Match'].notnull()].sort_values('Match',ascending=False).reset_index(drop=True)
            all_matches = pd.concat([all_matches,cy_data],ignore_index=True)
        except:
            errors.append(account)
    
    all_matches.drop_duplicates(inplace=True,ignore_index=True)
    all_matches['Match'] = all_matches['Match'].astype('int64')
        
    return all_matches, errors

In [37]:
all_matches, errors = get_player_matches(players['account_id'].values) 

100%|████████████████████████████████████████████████████████████████████████████████| 482/482 [05:29<00:00,  1.46it/s]


In [73]:
len(all_matches)
all_matches.sort_values('Match',inplace=True)

In [39]:
all_matches.to_csv('top_player_matches_8-26-2022.csv',index=False)

In [70]:
old_matches = pd.read_csv('top_player_matches_7-24-2022.csv').sort_values('Match')

In [71]:
set1 = set(old_matches['Match'].values)
set2 = set(all_matches['Match'].values)

In [75]:
len(set2.difference(set1))

955

In [47]:
len(diff)

10348

In [76]:
old_matches[old_matches['Match']==6707633683]

,Match


In [77]:
all_matches[all_matches['Match']==6707633683]

,Match
2,6707633683


In [ ]:
def add_recent_matches(league_id):
    response = requests.get(f'https://api.opendota.com/api/leagues/{league_id}/teams')